# Daily Market Data Update

Notebook operativo per aggiornamento incrementale OHLCV: seed Kaggle locali, API incremental, controlli DB e mini report.

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
for candidate in [PROJECT_ROOT, PROJECT_ROOT / 'src']:
    if str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from research_platform_core.data_platform import resolve_data_platform_roots, utc_now
from research_platform_core.loaders.kaggle_seed_loader import import_kaggle_seeds
from research_platform_core.ohlcv_ingest import OhlcvIngestJob, summarize_ohlcv_manifest
from research_platform_core.ohlcv_store import OhlcvDatabase

roots = resolve_data_platform_roots(repo_output_root=PROJECT_ROOT / 'output')
MARKETS = ['us_all', 'europe_major', 'japan_major', 'global_etfs']
START_DATE = '2000-01-01'
EXECUTE = os.environ.get('EXECUTE_MARKETDATA_NOTEBOOK', 'false').lower() == 'true'
UPDATE_KAGGLE = os.environ.get('UPDATE_KAGGLE_SEEDS', 'false').lower() == 'true'
DATABASE_URL = os.environ.get('MARKETDATA_DATABASE_URL')
print({'execute': EXECUTE, 'update_kaggle': UPDATE_KAGGLE, 'updated_at': utc_now()})

## 1. Optional Kaggle Seed Refresh

Imposta `UPDATE_KAGGLE_SEEDS=true` solo quando hai aggiornato manualmente i CSV/ZIP sotto `Database Finanziario/Kaggle`.

In [ ]:
if UPDATE_KAGGLE:
    kaggle_manifest = import_kaggle_seeds(
        financial_db_root=roots.financial_db,
        output_root=roots.repo_output,
        database_url=DATABASE_URL,
        dry_run=not EXECUTE,
    )
else:
    manifest_path = roots.financial_db / 'MarketData' / 'OHLCV' / 'manifests' / 'kaggle_seed_manifest.csv'
    kaggle_manifest = pd.read_csv(manifest_path) if manifest_path.exists() else pd.DataFrame()
kaggle_manifest

## 2. DB Check

In [ ]:
db = OhlcvDatabase(database_url=DATABASE_URL, financial_db_root=roots.financial_db)
db.init_schema()

def read_sql(query):
    if db.is_sqlite:
        with db.connect() as conn:
            return pd.read_sql_query(query, conn)
    with db.connect() as conn:
        return pd.read_sql_query(query, conn)

asset_status = read_sql("SELECT type, country, COUNT(*) AS assets FROM asset_master GROUP BY type, country ORDER BY assets DESC")
daily_status = read_sql("SELECT source, COUNT(*) AS rows, MAX(date) AS max_date FROM price_ohlcv_daily GROUP BY source ORDER BY rows DESC")
intraday_status = read_sql("SELECT source, COUNT(*) AS rows, MAX(ts) AS max_ts FROM price_ohlcv_intraday_5m GROUP BY source ORDER BY rows DESC")
display(asset_status.head(20))
display(daily_status)
display(intraday_status)

## 3. Incremental OHLCV API Update

La pipeline usa `latest_daily_dates_by_provider_symbol`, quindi i seed Kaggle gia' importati fanno partire le API da T0+1 invece che dal 2000.

In [ ]:
job = OhlcvIngestJob(roots.financial_db, roots.repo_output, database_url=DATABASE_URL)
api_manifest = job.run_daily(
    markets=MARKETS,
    start_date=START_DATE,
    mode='incremental',
    max_assets=None,
    dry_run=not EXECUTE,
)
display(summarize_ohlcv_manifest(api_manifest))
display(api_manifest.head(50))

## 4. Mini Coverage Report

In [ ]:
daily_coverage = read_sql('''
SELECT a.country, a.exchange, a.type, p.source,
       COUNT(DISTINCT a.asset_id) AS assets,
       COUNT(*) AS rows,
       MIN(p.date) AS min_date,
       MAX(p.date) AS max_date
FROM price_ohlcv_daily p
JOIN asset_master a ON a.asset_id = p.asset_id
GROUP BY a.country, a.exchange, a.type, p.source
ORDER BY rows DESC
''')
crypto_daily = daily_coverage[daily_coverage['type'].astype(str).str.lower().eq('crypto')].copy()
crypto_5m = read_sql('''
SELECT a.exchange, a.provider_symbol, p.source,
       COUNT(*) AS rows,
       MIN(p.ts) AS min_ts,
       MAX(p.ts) AS max_ts
FROM price_ohlcv_intraday_5m p
JOIN asset_master a ON a.asset_id = p.asset_id
WHERE LOWER(a.type) = 'crypto'
GROUP BY a.exchange, a.provider_symbol, p.source
ORDER BY rows DESC
''')
report_dir = roots.repo_output / 'tables'
report_dir.mkdir(parents=True, exist_ok=True)
daily_coverage.to_csv(report_dir / 'Daily_marketdata_coverage.csv', index=False)
crypto_5m.to_csv(report_dir / 'Daily_crypto_5m_coverage.csv', index=False)
display(daily_coverage.head(30))
display(crypto_daily)
display(crypto_5m.head(30))

## Scheduling

Esempio cron:

```cron
30 19 * * 1-5 cd /path/to/research_platform_definitive && EXECUTE_MARKETDATA_NOTEBOOK=true jupyter nbconvert --to notebook --execute notebooks/daily_marketdata_update.ipynb --output daily_marketdata_update_ran.ipynb
```

Quando aggiorni manualmente i dump Kaggle, aggiungi `UPDATE_KAGGLE_SEEDS=true`.